In [0]:

import pyspark.sql.types as T
import pyspark.sql.functions as F

### Dim brands

In [0]:
catalog_name = "ecommerce"
brand_schema = T.StructType([
  T.StructField("brand_id", T.StringType(), True),
  T.StructField("brand_name", T.StringType(), True),
  T.StructField("category_code", T.StringType(), True)])

In [0]:
raw_data_path = '/Volumes/ecommerce/source_data/raw/brands/brands.csv'
df = spark.read.option('header', 'true').option('delimeter',',').schema(brand_schema).csv(raw_data_path)
df= df.withColumn('ingested_at',F.current_timestamp())\
    .withColumn('source_file',F.col('_metadata.file_path'))
display(df.show(5))

+--------+-----------+-------------+--------------------+--------------------+
|brand_id| brand_name|category_code|         ingested_at|         source_file|
+--------+-----------+-------------+--------------------+--------------------+
|    ACME|   AcmeTech|           CE|2026-09-17 11:40:...|dbfs:/Volumes/eco...|
|    NOVW|  NovaWave |           CE|2026-09-17 11:40:...|dbfs:/Volumes/eco...|
|    ZNTH|     Zenith|           CE|2026-09-17 11:40:...|dbfs:/Volumes/eco...|
|    BYTM|    ByteMax|           CE|2026-09-17 11:40:...|dbfs:/Volumes/eco...|
|    ECOT|    EcoTone|           CE|2026-09-17 11:40:...|dbfs:/Volumes/eco...|
+--------+-----------+-------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .saveAsTable(f"{catalog_name}.bronze.brz_brands")

In [0]:
%sql
use catalog ecommerce

### Dim category

In [0]:
category_schema = T.StructType([
  T.StructField("category_name", T.StringType(), True),
  T.StructField("category_code", T.StringType(), True)])

In [0]:
raw_data_path = '/Volumes/ecommerce/source_data/raw/category/category.csv'
df =spark.read.option('header','true').option('delimeter',',').schema(category_schema).csv(raw_data_path)
df= df.withColumn('ingested_at',F.current_timestamp())\
    .withColumn('source_file',F.col('_metadata.file_path'))
display(df.show(5))

df.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .saveAsTable(f"{catalog_name}.bronze.brz_category")

+-------------+--------------------+--------------------+--------------------+
|category_name|       category_code|         ingested_at|         source_file|
+-------------+--------------------+--------------------+--------------------+
|           ce|         Electronics|2026-09-17 11:42:...|dbfs:/Volumes/eco...|
|          app|             Apparel|2026-09-17 11:42:...|dbfs:/Volumes/eco...|
|          hnk|      Home & Kitchen|2026-09-17 11:42:...|dbfs:/Volumes/eco...|
|          bpc|Beauty & Personal...|2026-09-17 11:42:...|dbfs:/Volumes/eco...|
|          bks|               Books|2026-09-17 11:42:...|dbfs:/Volumes/eco...|
+-------------+--------------------+--------------------+--------------------+
only showing top 5 rows


## Dim Customers

In [0]:
customer_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("phone", T.StringType(), True),
    T.StructField("country_code", T.StringType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("state", T.StringType(), True)
])

In [0]:
raw_data_path = "/Volumes/ecommerce/source_data/raw/customers/customers.csv"
df = spark.read.option('header','true').option('delimeter',',').schema(customer_schema).csv(raw_data_path)
df= df.withColumn('ingested_at',F.current_timestamp())\
    .withColumn('source_file',F.col('_metadata.file_path'))
display(df.show(5))
df.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .saveAsTable(f"{catalog_name}.bronze.brz_customers")


+----------------+--------------+------------+---------+-----+--------------------+--------------------+
|     customer_id|         phone|country_code|  country|state|         ingested_at|         source_file|
+----------------+--------------+------------+---------+-----+--------------------+--------------------+
|CUST000000000001|917280033536.0|          IN|    India|   MH|2026-09-17 11:43:...|dbfs:/Volumes/eco...|
|CUST000000000002|619489725433.0|          AU|Australia|  VIC|2026-09-17 11:43:...|dbfs:/Volumes/eco...|
|CUST000000000003|919390066524.0|          IN|    India|   TN|2026-09-17 11:43:...|dbfs:/Volumes/eco...|
|CUST000000000004|917073741793.0|          IN|    India|   TN|2026-09-17 11:43:...|dbfs:/Volumes/eco...|
|CUST000000000005|618478772532.0|          AU|Australia|   WA|2026-09-17 11:43:...|dbfs:/Volumes/eco...|
+----------------+--------------+------------+---------+-----+--------------------+--------------------+
only showing top 5 rows


## Dim Date

In [0]:
date_schema = T.StructType([
    T.StructField("date", T.StringType(), True),
    T.StructField("year", T.StringType(), True),
    T.StructField("day_name",T.StringType(),True),
    T.StructField("quarter",T.StringType(),True),
    T.StructField("week_of_year",T.StringType(),True)])
raw_data_path = "/Volumes/ecommerce/source_data/raw/date/date.csv"
df = spark.read.option('header','true').option('delimeter',',').schema(date_schema).csv(raw_data_path)
df= df.withColumn('ingested_at',F.current_timestamp())\
    .withColumn('source_file',F.col('_metadata.file_path'))
display(df.show(5))
df.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .saveAsTable(f"{catalog_name}.bronze.brz_date")

+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|         ingested_at|         source_file|
+----------+----+--------+-------+------------+--------------------+--------------------+
|01-08-2025|2025|  friday|      3|         -31|2026-09-17 11:43:...|dbfs:/Volumes/eco...|
|02-08-2025|2025|SATURDAY|      3|         -31|2026-09-17 11:43:...|dbfs:/Volumes/eco...|
|03-08-2025|2025|  SUNDAY|      3|         -31|2026-09-17 11:43:...|dbfs:/Volumes/eco...|
|04-08-2025|2025|  MONDAY|      3|         -32|2026-09-17 11:43:...|dbfs:/Volumes/eco...|
|05-08-2025|2025| TUESDAY|      3|         -32|2026-09-17 11:43:...|dbfs:/Volumes/eco...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 5 rows


## Dim products

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.bronze.brz_products")

DataFrame[]

In [0]:
products_schema = T.StructType([
    T.StructField("product_id", T.StringType(), False),
    T.StructField("sku", T.StringType(), True),
    T.StructField("category_code", T.StringType(), True),
    T.StructField("brand_code", T.StringType(), True),
    T.StructField("color", T.StringType(), True),
    T.StructField("size", T.StringType(), True),
    T.StructField("material", T.StringType(), True),
    T.StructField("weight_grams", T.StringType(), True),  #datatype is string due to incoming data contain anamolies
    T.StructField("length_cm", T.StringType(), True),     #datatype is string due to incoming data contain anamolies
    T.StructField("width_cm", T.FloatType(), True),
    T.StructField("height_cm", T.FloatType(), True),
    T.StructField("rating_count", T.IntegerType(), True),
    T.StructField("file_name", T.StringType(), False),
    T.StructField("ingest_timestamp", T.TimestampType(), False)
])

# Load data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/products/*.csv"

df = spark.read.option("header", "true").option("delimiter", ",").schema(products_schema).csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_products)
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_products")     